# P4 — Generation TTS pour Audiobook

**Navigation** : [Index](../README.md) | [<< Precedent](04-10-Annotation-Prosodique.ipynb) | [Suivant >>](04-12-Compilation-Audio.ipynb)

**Epic #1028** | Pass 4 : Generation audio via Kokoro TTS (demo) / FishAudio S2-Pro (production)

## Objectifs

1. Charger les annotations prosodiques P3 et la configuration vocale P2
2. Mapper chaque segment au bon modele/voix TTS
3. Generer les fichiers audio pour les 14 segments du texte
4. Exporter les metadonnees de generation (durees, tailles, paths)


In [1]:
# Import guards - availability flags for external dependencies

try:
    from IPython.display import display, Audio, Image as IPImage
    IPYTHON_AVAILABLE = True
except ImportError:
    IPYTHON_AVAILABLE = False
    print(f'  IPython non disponible - certaines fonctionnalites seront limitees')

try:
    import openai
    OPENAI_AVAILABLE = True
except ImportError:
    OPENAI_AVAILABLE = False
    print(f'  openai non disponible - certaines fonctionnalites seront limitees')

try:
    import requests
    REQUESTS_AVAILABLE = True
except ImportError:
    REQUESTS_AVAILABLE = False
    print(f'  requests non disponible - certaines fonctionnalites seront limitees')


import json
import os
import time
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import Optional
import requests

# Paths
BASE_DIR = Path(".")
PROSODIC_FILE = BASE_DIR / "prosodic_output" / "prosodic_annotations.json"
VOICE_CONFIG = BASE_DIR / "voice_casting_output" / "voice_casting_config.json"
TTS_OUTPUT_DIR = BASE_DIR / "tts_output"
TTS_OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

KOKORO_URL = "http://localhost:8191"
KOKORO_API_KEY = os.getenv("KOKORO_API_KEY", "")
print(f"Output dir: {TTS_OUTPUT_DIR.resolve()}")
print(f"Prosodic annotations: {PROSODIC_FILE.exists()}")
print(f"Voice config: {VOICE_CONFIG.exists()}")
print(f"Kokoro API key: {'set' if KOKORO_API_KEY else 'NOT SET'}")

Output dir: D:\Dev\CoursIA\MyIA.AI.Notebooks\GenAI\Audio\04-Applications\tts_output
Prosodic annotations: True
Voice config: True
Kokoro API key: set


## Architecture TTS

| Modele | Statut | Qualite | Latence | Expressivite |
|--------|--------|---------|---------|-------------|
| **Kokoro** (local Docker) | Actif | Bonne | <2s | Tags base (speed, emotion) |
| **FishAudio S2-Pro** | Non deploye | Excellente | ~3s | 15000+ tags expressifs |
| **OpenAI TTS-1** | Cloud | Bonne | ~2s | 6 voix fixes |

Ce notebook utilise **Kokoro** pour la demo locale. Les tags expressifs FishAudio sont conserves dans les metadonnees pour generation production ultérieure.

In [2]:
def load_voice_config(path):
    """Load P2 voice casting configuration."""
    with open(path) as f:
        return json.load(f)

def load_prosodic_annotations(path):
    """Load P3 prosodic annotations."""
    with open(path) as f:
        return json.load(f)

def build_character_voice_map(voice_config):
    """Map character_id -> kokoro voice_id."""
    mapping = {}
    for char_id, char_data in voice_config["characters"].items():
        kokoro = char_data["voices"]["kokoro"]
        mapping[char_id] = {
            "name": char_data["name"],
            "kokoro_voice": kokoro["voice_id"],
            "kokoro_url": kokoro["service_url"],
            "fish_preset": char_data["voices"]["fishaudio_production"]["preset"],
        }
    # Narrator fallback
    mapping["narrateur"] = mapping.get("narrateur", {
        "name": "Narrateur",
        "kokoro_voice": "bf_isabella",
        "kokoro_url": KOKORO_URL,
        "fish_preset": "narrator_male",
    })
    return mapping

voice_config = load_voice_config(VOICE_CONFIG)
prosodic = load_prosodic_annotations(PROSODIC_FILE)
voice_map = build_character_voice_map(voice_config)

print(f"Characters mapped: {len(voice_map)}")
for cid, info in voice_map.items():
    print(f"  {cid}: {info['name']} -> Kokoro {info['kokoro_voice']}")

Characters mapped: 8
  elisabeth_rousset: Elisabeth Rousset -> Kokoro af_sky
  cornudet: Cornudet -> Kokoro bm_george
  comtesse: Comtesse de Breville -> Kokoro af_sarah
  comte: Comte de Breville -> Kokoro am_michael
  narrateur: Narrateur -> Kokoro bf_isabella
  carre_lamadon: Monsieur Carre-Lamadon -> Kokoro bm_george
  loiseau: Monsieur Loiseau -> Kokoro bm_lewis
  soeurs: Les deux soeurs -> Kokoro bf_emma


## Kokoro TTS — API locale

Le service Kokoro (Docker, port 8191) expose une API compatible OpenAI :

```
POST /v1/audio/speech
Content-Type: application/json

{"model": "kokoro", "voice": "af_sky", "input": "texte", "speed": 1.0}
```

Reponse : audio MP3 (128 kbps, 24000 Hz, mono).

Note : Les tags expressifs `[whisper]`, `[shout]` etc. sont ignores par Kokoro mais conserves pour FishAudio S2-Pro en production.

In [3]:
@dataclass
class TTSRequest:
    """Represents a TTS generation request for one segment."""
    seg_index: int
    seg_type: str  # "dialogue", "narration", "description"
    speaker: str
    text: str
    annotated_text: str
    kokoro_voice: str
    fish_preset: str
    tags: list = field(default_factory=list)
    output_file: str = ""
    duration_s: float = 0.0
    file_size_kb: float = 0.0
    status: str = "pending"

def kokoro_tts(text, voice_id, speed=1.0):
    """Generate audio via Kokoro TTS API."""
    payload = {
        "model": "kokoro",
        "voice": voice_id,
        "input": text,
        "speed": speed,
    }
    headers = {"Content-Type": "application/json"}
    if KOKORO_API_KEY:
        headers["Authorization"] = f"Bearer {KOKORO_API_KEY}"
    try:
        resp = requests.post(
            f"{KOKORO_URL}/v1/audio/speech",
            json=payload,
            headers=headers,
            timeout=30,
        )
        resp.raise_for_status()
        return resp.content
    except requests.RequestException as e:
        print(f"  TTS error for voice={voice_id}: {e}")
        return None

print("TTS client ready")
print(f"Kokoro endpoint: {KOKORO_URL}")

TTS client ready
Kokoro endpoint: http://localhost:8191


## Construction des segments a generer

Chaque segment (dialogue ou narration) est prepare avec :
- Le texte annote P3 (avec tags expressifs)
- La voix Kokoro correspondante depuis P2
- Le preset FishAudio pour reference production

In [4]:
def resolve_speaker_id(speaker_name, voice_map):
    """Resolve a speaker name to character_id."""
    name_lower = speaker_name.lower().replace(" ", "_")
    # Direct match
    if name_lower in voice_map:
        return name_lower
    # Partial match
    for cid in voice_map:
        if cid in name_lower or name_lower in cid:
            return cid
    # Default narrator
    return "narrateur"

tts_requests = []

# Process utterances (dialogues)
for utt in prosodic["utterances"]:
    char_id = resolve_speaker_id(utt["speaker"], voice_map)
    voice_info = voice_map.get(char_id, voice_map["narrateur"])
    tts_requests.append(TTSRequest(
        seg_index=utt["seg_index"],
        seg_type=utt["utterance_type"],
        speaker=utt["speaker"],
        text=utt["original_text"],
        annotated_text=utt["annotated_text"],
        kokoro_voice=voice_info["kokoro_voice"],
        fish_preset=voice_info["fish_preset"],
        tags=utt["tags_applied"],
    ))

# Process narration segments
for seg in prosodic["narration_segments"]:
    voice_info = voice_map["narrateur"]
    tts_requests.append(TTSRequest(
        seg_index=seg["seg_index"],
        seg_type=seg["seg_type"],
        speaker="Narrateur",
        text=seg["original_text"],
        annotated_text=seg["annotated_text"],
        kokoro_voice=voice_info["kokoro_voice"],
        fish_preset=voice_info["fish_preset"],
        tags=seg["tags_applied"],
    ))

# Sort by segment index
tts_requests.sort(key=lambda r: r.seg_index)

print(f"Total segments to generate: {len(tts_requests)}")
for req in tts_requests:
    print(f"  Seg {req.seg_index}: {req.seg_type:12s} | "
          f"{req.kokoro_voice:15s} | {req.speaker}")

Total segments to generate: 14
  Seg 0: narration    | bf_isabella     | Narrateur
  Seg 1: narration    | bf_isabella     | Narrateur
  Seg 2: dialogue     | bf_isabella     | Narrateur
  Seg 3: dialogue     | bm_george       | Cornudet
  Seg 4: narration    | bf_isabella     | Narrateur
  Seg 5: dialogue     | af_sky          | Elisabeth Rousset
  Seg 6: dialogue     | af_sarah        | Comtesse de Breville
  Seg 7: dialogue     | bf_isabella     | Narrateur
  Seg 8: description  | bf_isabella     | Narrateur
  Seg 9: dialogue     | bf_isabella     | Narrateur
  Seg 10: dialogue     | bf_isabella     | Narrateur
  Seg 11: narration    | bf_isabella     | Narrateur
  Seg 12: dialogue     | am_michael      | Comte Hubert de Breville
  Seg 13: dialogue     | af_sky          | Elisabeth Rousset


## Generation audio

Chaque segment est envoye a Kokoro TTS avec la voix correspondante. Les tags expressifs sont conserves dans le texte brut (Kokoro les ignore). Les fichiers audio sont sauvegardes en MP3 (format natif de Kokoro).

In [5]:
import struct

def audio_duration_s(audio_bytes):
    """Extract duration from audio bytes (WAV or MP3)."""
    if len(audio_bytes) < 12:
        return 0.0
    # WAV format (RIFF header)
    if audio_bytes[:4] == b'RIFF':
        channels = struct.unpack_from('<H', audio_bytes, 22)[0]
        sample_rate = struct.unpack_from('<I', audio_bytes, 24)[0]
        bits_per_sample = struct.unpack_from('<H', audio_bytes, 34)[0]
        data_size = struct.unpack_from('<I', audio_bytes, 40)[0]
        bytes_per_sample = bits_per_sample * channels // 8
        if bytes_per_sample == 0:
            return 0.0
        return data_size / (sample_rate * bytes_per_sample)
    # MP3 format — estimate from file size at 128 kbps (Kokoro default)
    if audio_bytes[:3] == b'ID3' or audio_bytes[:2] == b'\xff\xfb':
        bitrate = 128000
        return len(audio_bytes) * 8 / bitrate
    return 0.0

generated = []
t_start = time.time()

for req in tts_requests:
    # Use original text for Kokoro (strip tags for cleaner output)
    clean_text = req.text
    output_name = f"seg_{req.seg_index:02d}_{req.kokoro_voice}.mp3"
    output_path = TTS_OUTPUT_DIR / output_name

    print(f"Generating seg {req.seg_index}: {req.kokoro_voice}...", end=" ")
    audio_data = kokoro_tts(clean_text, req.kokoro_voice)

    if audio_data:
        with open(output_path, "wb") as f:
            f.write(audio_data)
        req.output_file = str(output_path)
        req.duration_s = audio_duration_s(audio_data)
        req.file_size_kb = len(audio_data) / 1024
        req.status = "success"
        print(f"OK ({req.duration_s:.1f}s, {req.file_size_kb:.0f}KB)")
    else:
        req.status = "failed"
        print("FAILED")

    generated.append(req)
    time.sleep(0.2)  # Rate limit

elapsed = time.time() - t_start
success_count = sum(1 for r in generated if r.status == "success")
print(f"\nGenerated {success_count}/{len(generated)} segments in {elapsed:.1f}s")

Generating seg 0: bf_isabella... 

OK (9.0s, 140KB)
Generating seg 1: bf_isabella... 

OK (6.6s, 102KB)
Generating seg 2: bf_isabella... 

OK (6.8s, 107KB)
Generating seg 3: bm_george... 

OK (5.5s, 87KB)
Generating seg 4: bf_isabella... 

...

OK (4.6s, 72KB)
Generating seg 10: bf_isabella... 

OK (6.2s, 97KB)
Generating seg 11: bf_isabella... 

OK (14.7s, 229KB)
Generating seg 12: am_michael... 

OK (8.8s, 137KB)
Generating seg 13: af_sky... 

OK (7.8s, 122KB)

Generated 14/14 segments in 39.6s


## Resultats de la generation

Tableau recapitulatif de tous les segments generes.

In [6]:
print(f"{'Seg':>4} | {'Type':12s} | {'Voice':15s} | {'Speaker':25s} | "
      f"{'Duration':>8s} | {'Size':>7s} | {'Status':>7s}")
print("-" * 100)
total_duration = 0.0
for req in generated:
    dur = f"{req.duration_s:.1f}s" if req.status == "success" else "N/A"
    size = f"{req.file_size_kb:.0f}KB" if req.status == "success" else "N/A"
    print(f"{req.seg_index:>4} | {req.seg_type:12s} | {req.kokoro_voice:15s} | "
          f"{req.speaker:25s} | {dur:>8s} | {size:>7s} | {req.status:>7s}")
    if req.status == "success":
        total_duration += req.duration_s

print(f"\nTotal duration: {total_duration:.1f}s ({total_duration/60:.1f}min)")
print(f"Output directory: {TTS_OUTPUT_DIR.resolve()}")

 Seg | Type         | Voice           | Speaker                   | Duration |    Size |  Status
----------------------------------------------------------------------------------------------------
   0 | narration    | bf_isabella     | Narrateur                 |     9.0s |   140KB | success
   1 | narration    | bf_isabella     | Narrateur                 |     6.6s |   102KB | success
   2 | dialogue     | bf_isabella     | Narrateur                 |     6.8s |   107KB | success
   3 | dialogue     | bm_george       | Cornudet                  |     5.5s |    87KB | success
   4 | narration    | bf_isabella     | Narrateur                 |    19.1s |   299KB | success
   5 | dialogue     | af_sky          | Elisabeth Rousset         |     6.4s |   100KB | success
   6 | dialogue     | af_sarah        | Comtesse de Breville      |     5.8s |    90KB | success
   7 | dialogue     | bf_isabella     | Narrateur                 |     5.5s |    85KB | success
   8 | description  | bf_i

## Export des metadonnees

Les metadonnees de generation sont exportees en JSON pour P5 (Compilation audio).

In [7]:
tts_metadata = {
    "epic": "1028",
    "pass": "P4",
    "description": "TTS generation via Kokoro for audiobook demo",
    "source_text": "Boule de Suif - Guy de Maupassant",
    "tts_model": "kokoro",
    "tts_service_url": KOKORO_URL,
    "stats": {
        "total_segments": len(generated),
        "successful": sum(1 for r in generated if r.status == "success"),
        "failed": sum(1 for r in generated if r.status == "failed"),
        "total_duration_s": total_duration,
    },
    "segments": [asdict(r) for r in generated],
}

meta_path = TTS_OUTPUT_DIR / "tts_generation_metadata.json"
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(tts_metadata, f, ensure_ascii=False, indent=2)

print(f"Metadata saved: {meta_path}")
print(f"Segments: {tts_metadata['stats']['successful']}/{tts_metadata['stats']['total_segments']}")
print(f"Total audio: {total_duration:.1f}s")

Metadata saved: tts_output\tts_generation_metadata.json
Segments: 14/14
Total audio: 122.5s


## Analyse de l'utilisation des voix

Repartition des segments par voix et par type.

In [8]:
from collections import Counter

voice_usage = Counter(r.kokoro_voice for r in generated if r.status == "success")
type_usage = Counter(r.seg_type for r in generated if r.status == "success")

print("Voice usage:")
for voice, count in voice_usage.most_common():
    dur = sum(r.duration_s for r in generated if r.kokoro_voice == voice and r.status == "success")
    print(f"  {voice:20s}: {count} segments, {dur:.1f}s")

print(f"\nSegment types:")
for stype, count in type_usage.most_common():
    print(f"  {stype:12s}: {count}")

Voice usage:
  bf_isabella         : 9 segments, 88.2s
  af_sky              : 2 segments, 14.2s
  bm_george           : 1 segments, 5.5s
  af_sarah            : 1 segments, 5.8s
  am_michael          : 1 segments, 8.8s

Segment types:
  dialogue    : 9
  narration   : 4
  description : 1


## Conclusion P4 — Kokoro (Demo)

Les fichiers audio MP3 ont ete generes pour chaque segment du texte via Kokoro TTS. Les metadonnees (durees, voix, paths) sont exportees pour la compilation finale P5.

**Production** : Pour une qualite superieure, les memes segments sont generes ci-dessous via FishAudio S2-Pro avec les tags expressifs complets (`[whisper]`, `[shout]`, `[cold]`, etc.).

---

## P4b — Generation FishAudio S2-Pro (Production)

FishAudio S2-Pro (4.56B params, Dual-AR) genere un audio naturel avec support de 15000+ tags expressifs via la syntaxe `(emotion)`. Le service tourne en Docker sur GPU1 (RTX 3090 24GB).

In [9]:
FISHAUDIO_URL = os.getenv("FISHAUDIO_URL", "http://localhost:8197")

def fishaudio_tts(text, reference_id=None, timeout=300):
    """Generate audio via FishAudio S2-Pro API.

    Args:
        text: Text with optional emotion tags: (whispering), (shouting), (laughing)...
        reference_id: Optional voice reference ID for voice cloning
        timeout: Request timeout in seconds (300s for long texts)
    """
    payload = {"text": text}
    if reference_id:
        payload["reference_id"] = reference_id
    try:
        resp = requests.post(
            f"{FISHAUDIO_URL}/v1/tts",
            json=payload,
            timeout=timeout,
        )
        resp.raise_for_status()
        return resp.content
    except requests.RequestException as e:
        print(f"FishAudio TTS error: {e}")
        return None

def check_fishaudio_health(max_wait=120):
    """Check FishAudio health with retry (single-threaded service blocks during TTS)."""
    deadline = time.time() + max_wait
    while time.time() < deadline:
        try:
            resp = requests.get(f"{FISHAUDIO_URL}/v1/health", timeout=10)
            if resp.status_code == 200:
                return True
        except requests.RequestException:
            pass
        time.sleep(5)
    return False

def split_text_for_fishaudio(text, max_words=60):
    """Split long text into chunks of ~max_words at sentence boundaries."""
    words = text.split()
    if len(words) <= max_words:
        return [text]
    chunks = []
    current = []
    for word in words:
        current.append(word)
        if len(current) >= max_words and word.endswith(('.', '!', '?', ',', ';')):
            chunks.append(' '.join(current))
            current = []
    if current:
        chunks.append(' '.join(current))
    return chunks

def concatenate_wav_files(wav_files, output_path, silence_duration=0.3):
    """Concatenate multiple WAV files with silence gaps between them."""
    import io
    all_audio = b""
    sample_rate = 44100
    silence_samples = int(sample_rate * silence_duration)
    silence_bytes = b'\x00\x00' * silence_samples

    for i, wav_path in enumerate(wav_files):
        if isinstance(wav_path, (str, Path)):
            with open(wav_path, 'rb') as f:
                data = f.read()
        else:
            data = wav_path
        # Strip WAV header (44 bytes) and keep PCM data
        if data[:4] == b'RIFF':
            pcm_data = data[44:]
        else:
            pcm_data = data
        all_audio += pcm_data
        if i < len(wav_files) - 1:
            all_audio += silence_bytes

    # Build WAV header
    data_size = len(all_audio)
    header = struct.pack('<4sI4s', b'RIFF', 36 + data_size, b'WAVE')
    header += struct.pack('<4sIHHIIHH', b'fmt ', 16, 1, 1, sample_rate,
                          sample_rate * 2, 2, 16)
    header += struct.pack('<4sI', b'data', data_size)

    with open(output_path, 'wb') as f:
        f.write(header + all_audio)
    return output_path

fishaudio_available = check_fishaudio_health(max_wait=60)
if fishaudio_available:
    print(f"FishAudio S2-Pro: DISPONIBLE ({FISHAUDIO_URL})")
    print(f"Modele: 4.56B params Dual-AR, GPU1 RTX 3090")
    print(f"Timeout: 300s | Text split: 60 mots max")
else:
    print(f"FishAudio S2-Pro: NON DISPONIBLE ({FISHAUDIO_URL})")
    print("Le service FishAudio est optionnel — la generation Kokoro reste disponible.")

FishAudio S2-Pro: DISPONIBLE (http://localhost:8197)Modele: 4.56B params Dual-AR, GPU1 RTX 3090

In [10]:
# Tag mapping: P3 bracket notation -> FishAudio parenthesis notation
TAG_MAP = {
    "whisper": "whispering",
    "shout": "shouting",
    "scream": "screaming",
    "laugh": "laughing",
    "pause": None,
    "breath": "sighing",
    "cold": "(cold tone) ",
    "timid": "(timid tone) ",
    "angry": "(angry tone) ",
    "sad": "(sad tone) ",
    "excited": "(excitedly) ",
    "nervous": "(nervously) ",
    "gentle": "(gently) ",
    "firm": "(firmly) ",
    "mocking": "(mockingly) ",
    "whispered": "whispering",
    "shouted": "shouting",
}

def convert_tags_for_fishaudio(annotated_text):
    """Convert P3 bracket tags to FishAudio parenthesis tags."""
    import re
    result = annotated_text
    for bracket_tag, fish_tag in TAG_MAP.items():
        if fish_tag is None:
            # Remove pause/breath tags — handled by prosody
            result = result.replace(f"[{bracket_tag}]", "")
        elif fish_tag.startswith("("):
            # Already in parenthesis format
            result = result.replace(f"[{bracket_tag}]", fish_tag)
        else:
            # Convert to FishAudio format
            result = result.replace(f"[{bracket_tag}]", f"({fish_tag})")
    # Clean up double spaces from removed tags
    result = re.sub(r'\s+', ' ', result).strip()
    return result

print(f"Tag mapping: {len(TAG_MAP)} tags")
print(f"Exemple: '[whisper] Bonjour' -> '{convert_tags_for_fishaudio('[whisper] Bonjour')}'")
print(f"Exemple: '[shout] Au revoir' -> '{convert_tags_for_fishaudio('[shout] Au revoir')}'")
print(f"Exemple: '[cold] Je vois' -> '{convert_tags_for_fishaudio('[cold] Je vois')}'")

Tag mapping: 17 tagsExemple: '[whisper] Bonjour' -> '(whispering) Bonjour'Exemple: '[shout] Au revoir' -> '(shouting) Au revoir'Exemple: '[cold] Je vois' -> '(cold tone) Je vois'

## Decoupage intelligent des segments (LLM)

Les segments longs (>55 mots) doivent etre decoupes pour FishAudio. Un decoupage naif par comptage de mots coupe au milieu d'une phrase ou d'une scene. Cette etape utilise un LLM pour decouper aux **frontieres naturelles** du texte : fins de phrase, changements de scene, transitions narratif/dialogue.

La fonction `llm_split_segment()` envoie le texte a `gpt-4o-mini` qui identifie les points de coupure optimaux, avec un fallback sur le decoupage par phrase si le LLM est indisponible.

In [11]:
from openai import OpenAI

openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))


def llm_split_segment(text, max_words=55):
    """Split a long text at natural narrative boundaries using an LLM.

    Falls back to sentence-based splitting if the LLM is unavailable.
    """
    words = text.split()
    if len(words) <= max_words:
        return [text]

    prompt = (
        "Decoupe le texte suivant en 2 a 3 parties pour la synthese vocale.\n"
        "Chaque partie doit avoir un sens narratif complet (fin de phrase, fin de scene).\n"
        "Conserve exactement le meme texte, coupes aux positions naturelles.\n"
        "Ne reformule rien. Respecte les balises emotion (parentheses).\n"
        f"Texte: {text}\n"
        "Reponds UNIQUEMENT avec les parties separees par |||"
    )
    try:
        resp = openai_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.1,
            max_tokens=1000,
        )
        parts = resp.choices[0].message.content.strip().split("|||")
        parts = [p.strip() for p in parts if p.strip()]
        if len(parts) >= 2:
            print(f"  LLM split: {len(words)} mots -> {len(parts)} parties")
            return parts
    except Exception as e:
        print(f"  LLM split fallback (error: {e})")

    return split_text_for_fishaudio(text, max_words=max_words)


# Test sur un segment long
test_long = "La diligence partit dans la neige. Dans la voiture, neuf passagers se tenaient serres : le marchand de vins Loiseau et sa femme, le republican Cornudet, la comtesse de Breville, le manufacturier Carre-Lamadon et son epouse, le comte Hubert de Breville, les bonnes soeurs, et Elisabeth Rousset, dite Boule de Suif, une prostituee de grand renom."
parts = llm_split_segment(test_long)
for i, p in enumerate(parts):
    print(f"  Part {i+1} ({len(p.split())} mots): {p[:60]}...")

  LLM split: 56 mots -> 2 parties  Part 1 (31 mots): La diligence partit dans la neige. Dans la voiture, neuf pas...  Part 2 (25 mots): le marchand de vins Loiseau et sa femme, le republican Co...

In [12]:
FISH_OUTPUT_DIR = BASE_DIR / "tts_output_fishaudio"
FISH_OUTPUT_DIR.mkdir(exist_ok=True, parents=True)
SUB_DIR = FISH_OUTPUT_DIR / "sub_segments"
SUB_DIR.mkdir(exist_ok=True)

fish_generated = []

if fishaudio_available:
    fish_t_start = time.time()

    for req in tts_requests:
        fish_text = convert_tags_for_fishaudio(req.annotated_text)
        output_name = f"fish_seg_{req.seg_index:02d}_{req.fish_preset}.wav"
        output_path = FISH_OUTPUT_DIR / output_name

        # Skip if already successfully generated (>10KB = valid audio)
        if output_path.exists() and output_path.stat().st_size > 10000:
            existing_data = output_path.read_bytes()
            dur = audio_duration_s(existing_data)
            fish_generated.append(TTSRequest(
                seg_index=req.seg_index, seg_type=req.seg_type,
                speaker=req.speaker, text=req.text,
                annotated_text=fish_text, kokoro_voice=req.kokoro_voice,
                fish_preset=req.fish_preset, tags=req.tags,
                output_file=str(output_path), duration_s=dur,
                file_size_kb=len(existing_data) / 1024, status="success",
            ))
            print(f"Seg {req.seg_index:2d} ({req.seg_type:12s}): CACHED ({dur:.1f}s)")
            continue

        # Use LLM-based splitting for natural narrative boundaries
        chunks = llm_split_segment(fish_text, max_words=55)
        print(f"Seg {req.seg_index:2d} ({req.seg_type:12s}, {req.speaker}): "
              f"{len(chunks)} chunk(s), {len(req.text.split())} mots")

        chunk_files = []
        all_ok = True
        max_retries = 2

        for ci, chunk in enumerate(chunks):
            chunk_name = f"sub_{req.seg_index:02d}_{ci}_{req.fish_preset}.wav"
            chunk_path = SUB_DIR / chunk_name

            audio_data = None
            for attempt in range(max_retries):
                print(f"  Chunk {ci+1}/{len(chunks)} ({len(chunk.split())} mots)"
                      f" [attempt {attempt+1}]...", end=" ", flush=True)
                audio_data = fishaudio_tts(chunk, timeout=300)

                if audio_data and len(audio_data) > 1000:
                    with open(chunk_path, "wb") as f:
                        f.write(audio_data)
                    chunk_files.append(chunk_path)
                    dur_chunk = audio_duration_s(audio_data)
                    print(f"OK ({dur_chunk:.1f}s, {len(audio_data)/1024:.0f}KB)")
                    break
                else:
                    print(f"FAILED (got {len(audio_data) if audio_data else 0} bytes)")
                    if attempt < max_retries - 1:
                        print(f"    Retrying in 5s...")
                        time.sleep(5)

            if not audio_data or len(audio_data) <= 1000:
                all_ok = False
                break

            # Pause between chunks (FishAudio is single-threaded, no healthcheck needed)
            if ci < len(chunks) - 1:
                time.sleep(3)

        if all_ok and chunk_files:
            import shutil
            if len(chunk_files) == 1:
                shutil.copy2(chunk_files[0], output_path)
            else:
                concatenate_wav_files(chunk_files, output_path)

            final_data = output_path.read_bytes()
            final_dur = audio_duration_s(final_data)
            fish_generated.append(TTSRequest(
                seg_index=req.seg_index, seg_type=req.seg_type,
                speaker=req.speaker, text=req.text,
                annotated_text=fish_text, kokoro_voice=req.kokoro_voice,
                fish_preset=req.fish_preset, tags=req.tags,
                output_file=str(output_path), duration_s=final_dur,
                file_size_kb=len(final_data) / 1024, status="success",
            ))
            print(f"  => Seg {req.seg_index}: SUCCESS ({final_dur:.1f}s, {len(final_data)/1024:.0f}KB)")
        else:
            fish_generated.append(TTSRequest(
                seg_index=req.seg_index, seg_type=req.seg_type,
                speaker=req.speaker, text=req.text,
                annotated_text=fish_text, kokoro_voice=req.kokoro_voice,
                fish_preset=req.fish_preset, tags=req.tags,
                status="failed",
            ))
            print(f"  => Seg {req.seg_index}: FAILED")

        # Pause between segments
        time.sleep(2)

    fish_elapsed = time.time() - fish_t_start
    fish_ok = sum(1 for r in fish_generated if r.status == "success")
    print(f"\n{'='*60}")
    print(f"FishAudio: {fish_ok}/{len(fish_generated)} segments in {fish_elapsed:.0f}s")
else:
    print("FishAudio S2-Pro non disponible — generation skippee")
    print("Demarrer le service: docker compose -f docker-configurations/services/tts-fishaudio/docker-compose.yml up -d")

Seg  0 (narration   ): CACHED (7.9s)Seg  1 (narration   ): CACHED (7.1s)Seg  2 (dialogue    ): CACHED (5.7s)Seg  3 (dialogue    ): CACHED (4.1s)Seg  4 (narration   ): CACHED (23.6s)Seg  5 (dialogue    ): CACHED (6.8s)Seg  6 (dialogue    ): CACHED (4.8s)Seg  7 (dialogue    ): CACHED (4.1s)Seg  8 (description ): CACHED (16.7s)Seg  9 (dialogue    ): CACHED (5.8s)Seg 10 (dialogue    ): CACHED (7.0s)Seg 11 (narration   ): CACHED (17.3s)Seg 12 (dialogue    ): CACHED (10.4s)Seg 13 (dialogue    ): CACHED (6.6s)============================================================FishAudio: 14/14 segments in 5s

In [13]:
if fish_generated:
    kokoro_dur = sum(r.duration_s for r in generated if r.status == "success")
    fish_dur = sum(r.duration_s for r in fish_generated if r.status == "success")
    kokoro_size = sum(r.file_size_kb for r in generated if r.status == "success")
    fish_size = sum(r.file_size_kb for r in fish_generated if r.status == "success")

    print("Comparaison Kokoro vs FishAudio S2-Pro:")
    print(f"  {'':20s} | {'Kokoro':>10s} | {'FishAudio':>10s}")
    print(f"  {'Segments':20s} | {len(generated):>10d} | {len(fish_generated):>10d}")
    print(f"  {'Duree totale':20s} | {kokoro_dur:>9.1f}s | {fish_dur:>9.1f}s")
    print(f"  {'Taille totale':20s} | {kokoro_size:>8.0f}KB | {fish_size:>8.0f}KB")
    print(f"  {'Duree moy/seg':20s} | {kokoro_dur/len(generated):>9.1f}s | {fish_dur/max(len(fish_generated),1):>9.1f}s")
    print(f"  {'Format audio':20s} | {'MP3':>10s} | {'WAV':>10s}")
    print(f"  {'Tags expressifs':20s} | {'Non':>10s} | {'Oui (15000+)':>10s}")
else:
    print("FishAudio non disponible — comparaison impossible")
    print("Le service FishAudio S2-Pro doit tourner sur le port 8197")

Comparaison Kokoro vs FishAudio S2-Pro:                        |     Kokoro |  FishAudio  Segments             |         14 |         14  Duree totale         |     122.5s |     127.9s  Taille totale        |     1913KB |    10988KB  Duree moy/seg        |       8.7s |       9.1s  Format audio         |        MP3 |        WAV  Tags expressifs      |        Non | Oui (15000+)

In [14]:
# Export FishAudio generation metadata for P5
fish_meta = {
    "epic": "1273",
    "pass": "P4b",
    "description": "TTS generation via FishAudio S2-Pro for audiobook production",
    "source_text": "Boule de Suif - Guy de Maupassant",
    "tts_model": "fishaudio-s2-pro",
    "tts_service_url": FISHAUDIO_URL,
    "stats": {
        "total_segments": len(tts_requests),
        "fish_generated": len(fish_generated),
        "fish_successful": sum(1 for r in fish_generated if r.status == "success"),
        "fish_total_duration_s": sum(r.duration_s for r in fish_generated if r.status == "success"),
    },
    "segments": [asdict(r) for r in fish_generated],
}

fish_meta_path = FISH_OUTPUT_DIR / "fish_generation_metadata.json"
with open(fish_meta_path, "w", encoding="utf-8") as f:
    json.dump(fish_meta, f, ensure_ascii=False, indent=2)

print(f"FishAudio metadata saved: {fish_meta_path}")
print(f"Segments: {fish_meta['stats']['fish_successful']}/{fish_meta['stats']['total_segments']}")
if fish_generated:
    fish_dur = fish_meta['stats']['fish_total_duration_s']
    print(f"Total FishAudio audio: {fish_dur:.1f}s ({fish_dur/60:.1f}min)")
else:
    print("No FishAudio segments generated")

FishAudio metadata saved: tts_output_fishaudioish_generation_metadata.jsonSegments: 14/14Total FishAudio audio: 127.9s (2.1min)

## Compilation du livre audio complet

Tous les segments FishAudio S2-Pro generes sont concatenes en un seul fichier WAV pour obtenir le livre audio complet. Des silences de 0.5s sont inseres entre les segments pour marquer les transitions narratif/dialogue.

In [15]:
if fish_generated:
    success_segments = sorted(
        [r for r in fish_generated if r.status == "success"],
        key=lambda r: r.seg_index
    )
    if len(success_segments) >= 10:
        wav_files = [r.output_file for r in success_segments]
        audiobook_path = FISH_OUTPUT_DIR / "boule_de_suif_fishaudio_complete.wav"
        concatenate_wav_files(wav_files, audiobook_path, silence_duration=0.5)

        ab_data = audiobook_path.read_bytes()
        ab_dur = audio_duration_s(ab_data)
        print(f"Livre audio complet: {audiobook_path.name}")
        print(f"  Duree: {ab_dur:.1f}s ({ab_dur/60:.1f}min)")
        print(f"  Taille: {len(ab_data)/1024:.0f}KB")
        print(f"  Segments: {len(success_segments)}/{len(fish_generated)}")
        print(f"  Silences: 0.5s entre segments")
        print()

        for r in success_segments:
            dur_str = f"{r.duration_s:.1f}s"
            print(f"  Seg {r.seg_index:2d} | {r.seg_type:12s} | {r.speaker:25s} | {dur_str}")
    else:
        print(f"Seulement {len(success_segments)}/{len(fish_generated)} segments OK")
        print("Audiobook non compile — generation incomplete")
else:
    print("Aucun segment FishAudio genere — audiobook non disponible")

Livre audio complet: boule_de_suif_fishaudio_complete.wav  Duree: 134.4s (2.2min)  Taille: 11576KB  Segments: 14/14  Silences: 0.5s entre segments  Seg  0 | narration    | Narrateur                 | 7.9s  Seg  1 | narration    | Narrateur                 | 7.1s  Seg  2 | dialogue     | Narrateur                 | 5.7s  Seg  3 | dialogue     | Cornudet                  | 4.1s  Seg  4 | narration    | Narrateur                 | 23.6s  Seg  5 | dialogue     | Elisabeth Rousset         | 6.8s  Seg  6 | dialogue     | Comtesse de Breville      | 4.8s  Seg  7 | dialogue     | Narrateur                 | 4.1s  Seg  8 | description  | Narrateur                 | 16.7s  Seg  9 | dialogue     | Narrateur                 | 5.8s  Seg 10 | dialogue     | Narrateur                 | 7.0s  Seg 11 | narration    | Narrateur                 | 17.3s  Seg 12 | dialogue     | Comte Hubert de Breville  | 10.4s  Seg 13 | dialogue     | Elisabeth Rousset         | 6.6s

## Comparaison detaillee Kokoro vs FishAudio (segment par segment)

Pour chaque segment, vous pouvez ecouter les deux versions et comparer la qualite, l'expressivite et le naturel de la synthese.

In [16]:
from IPython.display import Audio, HTML, display
import json

# Load Kokoro metadata
kokoro_meta_path = BASE_DIR / "tts_output" / "tts_generation_metadata.json"
fish_meta_path = FISH_OUTPUT_DIR / "fish_generation_metadata.json"

if kokoro_meta_path.exists() and fish_meta_path.exists():
    with open(kokoro_meta_path) as f:
        kokoro_meta = json.load(f)
    with open(fish_meta_path) as f:
        fish_meta = json.load(f)

    kokoro_segs = {s["seg_index"]: s for s in kokoro_meta["segments"]}
    fish_segs = {s["seg_index"]: s for s in fish_meta["segments"]}

    print("=" * 80)
    print(f"{'Seg':>3} | {'Type':12s} | {'Kokoro':>8s} | {'FishAudio':>8s} | {'Delta':>7s} | "
          f"{'Kokoro KB':>9s} | {'Fish KB':>8s} | {'Speaker':25s}")
    print("-" * 110)

    total_kokoro = 0.0
    total_fish = 0.0
    for i in range(14):
        ks = kokoro_segs.get(i, {})
        fs = fish_segs.get(i, {})
        k_dur = ks.get("duration_s", 0)
        f_dur = fs.get("duration_s", 0)
        k_size = ks.get("file_size_kb", 0)
        f_size = fs.get("file_size_kb", 0)
        delta = f_dur - k_dur
        seg_type = ks.get("seg_type", "?")
        speaker = ks.get("speaker", "?")

        delta_str = f"+{delta:.1f}s" if delta >= 0 else f"{delta:.1f}s"
        k_status = "OK" if ks.get("status") == "success" else "FAIL"
        f_status = "OK" if fs.get("status") == "success" else "FAIL"

        print(f"{i:>3} | {seg_type:12s} | {k_dur:>7.1f}s | {f_dur:>7.1f}s | {delta_str:>7s} | "
              f"{k_size:>8.0f}KB | {f_size:>6.0f}KB | {speaker:25s}")
        total_kokoro += k_dur
        total_fish += f_dur

    print("-" * 110)
    delta_total = total_fish - total_kokoro
    print(f"{'TOT':>3} | {'':12s} | {total_kokoro:>7.1f}s | {total_fish:>7.1f}s | {delta_total:>+7.1f}s |")
    print(f"\nKokoro total: {total_kokoro:.1f}s ({total_kokoro/60:.1f}min) | FishAudio total: {total_fish:.1f}s ({total_fish/60:.1f}min)")
    print(f"FishAudio est {abs(delta_total):.1f}s {'plus lent' if delta_total > 0 else 'plus rapide'} que Kokoro")

    # Interactive audio player for each segment
    print("\n" + "=" * 80)
    print("Ecoute comparative segment par segment :")
    print("=" * 80)

    for i in range(14):
        ks = kokoro_segs.get(i, {})
        fs = fish_segs.get(i, {})
        if ks.get("status") != "success" or fs.get("status") != "success":
            continue

        text = ks.get("text", "")[:80] + ("..." if len(ks.get("text", "")) > 80 else "")
        display(HTML(f"<h4>Segment {i} — {ks.get('seg_type','?')} ({ks.get('speaker','?')})</h4>"))
        display(HTML(f"<p><i>{text}</i></p>"))

        # Kokoro player
        kokoro_path = BASE_DIR / ks["output_file"]
        if kokoro_path.exists():
            display(HTML(f"<b>Kokoro</b> ({ks['duration_s']:.1f}s, {ks['kokoro_voice']})"))
            display(Audio(filename=str(kokoro_path)))

        # FishAudio player
        fish_path = BASE_DIR / fs["output_file"]
        if fish_path.exists():
            display(HTML(f"<b>FishAudio S2-Pro</b> ({fs['duration_s']:.1f}s, {fs['fish_preset']})"))
            display(Audio(filename=str(fish_path)))

        display(HTML("<hr>"))

    # Audiobook comparison
    print("=" * 80)
    print("Livres audio complets :")
    print("=" * 80)

    kokoro_audiobook = BASE_DIR / "audiobook_kokoro.mp3"
    fish_audiobook = BASE_DIR / "audiobook_fishaudio.wav"

    if kokoro_audiobook.exists():
        display(HTML("<h4>Livre audio complet — Kokoro</h4>"))
        display(Audio(filename=str(kokoro_audiobook)))
    if fish_audiobook.exists():
        display(HTML("<h4>Livre audio complet — FishAudio S2-Pro</h4>"))
        display(Audio(filename=str(fish_audiobook)))
else:
    print("Metadonnees manquantes — comparaison impossible")
    print(f"  Kokoro: {kokoro_meta_path} ({'OK' if kokoro_meta_path.exists() else 'MANQUANT'})")
    print(f"  FishAudio: {fish_meta_path} ({'OK' if fish_meta_path.exists() else 'MANQUANT'})")

Seg | Type         |   Kokoro | FishAudio |   Delta | Kokoro KB |  Fish KB | Speaker                  
--------------------------------------------------------------------------------------------------------------
  0 | narration    |     8.9s |     7.9s |   -1.1s |      140KB |    680KB | Narrateur                
  1 | narration    |     6.5s |     7.1s |   +0.5s |      102KB |    608KB | Narrateur                
  2 | dialogue     |     6.8s |     5.7s |   -1.2s |      107KB |    488KB | Narrateur                
  3 | dialogue     |     5.5s |     4.1s |   -1.4s |       87KB |    356KB | Cornudet                 
  4 | narration    |    19.1s |    23.6s |   +4.5s |      299KB |   2032KB | Narrateur                
  5 | dialogue     |     6.4s |     6.8s |   +0.4s |      100KB |    588KB | Elisabeth Rousset        
  6 | dialogue     |     5.8s |     4.8s |   -1.0s |       90KB |    416KB | Comtesse de Breville     
  7 | dialogue     |     5.5s |     4.1s |   -1.4s |       85KB |

Segment 0 — narration (Narrateur)

Pendant plusieurs jours, des lambeaux d'armees en deroute avaient traverse la ville...

Kokoro (8.9s, bf_isabella)

[Audio: seg_00_bf_isabella.mp3]

...

Livres audio complets :

Livre audio complet — Kokoro

[Audio: audiobook_kokoro.mp3]

Livre audio complet — FishAudio S2-Pro

[Audio: audiobook_fishaudio.wav]